# DatetimeOutputParser

`DatetimeOutputParser`는 LLM의 날짜/시간 출력을 Python의 `datetime` 형식으로 변환하는 출력 파서입니다.

즉, 자연어로 받은 날짜를 일정한 날짜 형식으로 정리해서 사용할 때 유용합니다.

In [1]:
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
# LangSmith 추적 설정
from langchain_teddynote import logging

logging.langsmith("CH03-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


## 날짜/시간 형식 코드

자주 사용하는 형식만 기억하면 됩니다.

| 코드 | 의미 | 예시 |
|---|---|---|
| `%Y` | 4자리 연도 | 2026 |
| `%m` | 월 | 09 |
| `%d` | 일 | 16 |
| `%H` | 24시간 기준 시 | 14 |
| `%M` | 분 | 30 |
| `%S` | 초 | 08 |

예를 들어:

`%Y-%m-%d` → `2026-09-16`

In [3]:
from langchain_classic.output_parsers import DatetimeOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 날짜 출력 파서 생성
output_parser = DatetimeOutputParser()

# 원하는 날짜 형식 지정
output_parser.format = "%Y-%m-%d"

# 사용자 질문용 프롬프트
template = """Answer the users question:

#Format Instructions:
{format_instructions}

#Question:
{question}

#Answer:"""

prompt = PromptTemplate.from_template(
    template,
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    },
)

# 프롬프트 확인
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={'format_instructions': "Write a datetime string that matches the following pattern: '%Y-%m-%d'.\n\nExamples: 2026-09-16, 2025-09-16, 2026-09-15\n\nReturn ONLY this string, no other words!"}, template='Answer the users question:\n\n#Format Instructions:\n{format_instructions}\n\n#Question:\n{question}\n\n#Answer:')

Prompt → ChatOpenAI → DatetimeOutputParser 순서로 체인을 연결합니다.

In [4]:
# 체인 생성
chain = prompt | ChatOpenAI(model="gpt-4.1-mini") | output_parser

# Google 창업 연도를 질문
output = chain.invoke({"question": "Google 이 창업한 연도"})

In [5]:
ChatOpenAI()

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-3.5-turbo', 'status': 'deprecated', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001BC2B8642C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BC2B7DA060>, root_client=<openai.Open

In [6]:
ChatOpenAI(model="gpt-4.1-mini")

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001BC1562A630>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BC2B9F4500>, root_client=<openai.OpenAI object at 0x000001BC1350E9C

파싱된 `datetime` 결과를 원하는 문자열 형식으로 다시 출력합니다.

In [7]:
# datetime 결과를 문자열로 변환
output.strftime("%Y-%m-%d")

'1998-09-04'